# UCS420 – Cognitive Computing | Assignment 10
## Diabetes Care Information Assistant

**Name:** Anmol Singh  
**Roll Number:** `1024170048`  
**Selected domain:** Diabetes Care Information Assistant (the roll number is odd).



## 1. Install and import libraries

In [ ]:
!pip -q install nltk scikit-learn pandas matplotlib

In [ ]:
import re, nltk, numpy as np, pandas as pd, matplotlib.pyplot as plt
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

for resource in ['punkt','punkt_tab','stopwords','wordnet','omw-1.4']:
    nltk.download(resource, quiet=True)
STOP_WORDS = set(stopwords.words('english'))
LEMMATIZER = WordNetLemmatizer()
print("Setup complete.")

## 2. Labelled corpus\nThe dataset contains 60 illustrative queries across six intent categories (10 per category).

In [ ]:
data = pd.DataFrame([{'query': 'What is diabetes?', 'intent': 'general_information'}, {'query': 'Explain diabetes mellitus.', 'intent': 'general_information'}, {'query': 'What happens in the body when someone has diabetes?', 'intent': 'general_information'}, {'query': 'What is the difference between type 1 and type 2 diabetes?', 'intent': 'general_information'}, {'query': 'Can you explain insulin resistance?', 'intent': 'general_information'}, {'query': 'What does it mean to have high blood sugar?', 'intent': 'general_information'}, {'query': 'Why is diabetes called a chronic condition?', 'intent': 'general_information'}, {'query': 'What are the common types of diabetes?', 'intent': 'general_information'}, {'query': 'Tell me the basics of diabetes.', 'intent': 'general_information'}, {'query': 'How does diabetes affect glucose levels?', 'intent': 'general_information'}, {'query': 'What are common symptoms of diabetes?', 'intent': 'symptoms'}, {'query': 'Why am I feeling very thirsty all the time?', 'intent': 'symptoms'}, {'query': 'Can frequent urination be a sign of diabetes?', 'intent': 'symptoms'}, {'query': 'Does diabetes cause blurry vision?', 'intent': 'symptoms'}, {'query': 'What are signs of low blood sugar?', 'intent': 'symptoms'}, {'query': 'What does high blood sugar feel like?', 'intent': 'symptoms'}, {'query': 'Can unexplained weight loss occur with diabetes?', 'intent': 'symptoms'}, {'query': 'Why do people with diabetes feel tired?', 'intent': 'symptoms'}, {'query': 'What are symptoms of hypoglycemia?', 'intent': 'symptoms'}, {'query': 'What are warning signs of hyperglycemia?', 'intent': 'symptoms'}, {'query': 'What foods should a person with diabetes choose?', 'intent': 'nutrition'}, {'query': 'Can people with diabetes eat fruit?', 'intent': 'nutrition'}, {'query': 'Explain carbohydrate counting.', 'intent': 'nutrition'}, {'query': 'Is whole grain food a better choice?', 'intent': 'nutrition'}, {'query': 'What is a balanced meal for diabetes?', 'intent': 'nutrition'}, {'query': 'How can I reduce added sugar in my diet?', 'intent': 'nutrition'}, {'query': 'Are sugary drinks suitable for people with diabetes?', 'intent': 'nutrition'}, {'query': 'What are healthy snack ideas for diabetes?', 'intent': 'nutrition'}, {'query': 'How does portion size affect blood glucose?', 'intent': 'nutrition'}, {'query': 'Should a diabetes meal plan include vegetables?', 'intent': 'nutrition'}, {'query': 'How is blood glucose measured?', 'intent': 'monitoring'}, {'query': 'What is a blood glucose meter?', 'intent': 'monitoring'}, {'query': 'When should someone check their blood sugar?', 'intent': 'monitoring'}, {'query': 'What does HbA1c measure?', 'intent': 'monitoring'}, {'query': 'Why is continuous glucose monitoring used?', 'intent': 'monitoring'}, {'query': 'How do I record glucose readings?', 'intent': 'monitoring'}, {'query': 'What is a fasting blood glucose test?', 'intent': 'monitoring'}, {'query': 'What is the purpose of an A1C test?', 'intent': 'monitoring'}, {'query': 'How can I understand glucose trends?', 'intent': 'monitoring'}, {'query': 'Why should glucose readings be tracked?', 'intent': 'monitoring'}, {'query': 'What is insulin used for?', 'intent': 'medication_and_treatment'}, {'query': 'What are common ways diabetes is managed?', 'intent': 'medication_and_treatment'}, {'query': 'Why might a doctor prescribe metformin?', 'intent': 'medication_and_treatment'}, {'query': 'What should I know about diabetes medicines?', 'intent': 'medication_and_treatment'}, {'query': 'Can lifestyle changes help manage type 2 diabetes?', 'intent': 'medication_and_treatment'}, {'query': 'Why is it important to take prescribed medicine as directed?', 'intent': 'medication_and_treatment'}, {'query': 'What is insulin therapy?', 'intent': 'medication_and_treatment'}, {'query': 'Who decides the dose of diabetes medication?', 'intent': 'medication_and_treatment'}, {'query': 'What should I do if I have questions about side effects?', 'intent': 'medication_and_treatment'}, {'query': 'Can exercise be part of diabetes management?', 'intent': 'medication_and_treatment'}, {'query': 'How can I lower my risk of type 2 diabetes?', 'intent': 'prevention_and_lifestyle'}, {'query': 'Does regular exercise help with diabetes risk?', 'intent': 'prevention_and_lifestyle'}, {'query': 'Why is maintaining a healthy weight important?', 'intent': 'prevention_and_lifestyle'}, {'query': 'Can walking support blood sugar management?', 'intent': 'prevention_and_lifestyle'}, {'query': 'What lifestyle habits support diabetes prevention?', 'intent': 'prevention_and_lifestyle'}, {'query': 'How much does sleep matter for metabolic health?', 'intent': 'prevention_and_lifestyle'}, {'query': 'Does smoking affect diabetes complications?', 'intent': 'prevention_and_lifestyle'}, {'query': 'How can stress management support health?', 'intent': 'prevention_and_lifestyle'}, {'query': 'What are healthy daily habits for diabetes?', 'intent': 'prevention_and_lifestyle'}, {'query': 'Can physical activity improve insulin sensitivity?', 'intent': 'prevention_and_lifestyle'}])
print("Queries:", len(data), "| Intent categories:", data.intent.nunique())
display(data.groupby('intent').size().to_frame('query_count'))

## 3. Hierarchical taxonomy
Four major categories, each with two subcategories.

In [ ]:
taxonomy = {
    "Understanding Diabetes": ["General information", "Types and causes"],
    "Recognising Health Signals": ["High blood glucose symptoms", "Low blood glucose symptoms"],
    "Daily Self-Management": ["Nutrition and meal planning", "Physical activity and lifestyle"],
    "Clinical Care and Tracking": ["Glucose monitoring and tests", "Medication and treatment"]
}
for major, subs in taxonomy.items():
    print(major)
    for sub in subs: print("  -", sub)

## 4. Domain concepts and relationships

In [ ]:
concepts = [
    "Diabetes", "Blood Glucose", "Insulin", "Pancreas", "Carbohydrates",
    "Nutrition", "Physical Activity", "Glucose Monitoring", "HbA1c",
    "Medication", "Hypoglycemia", "Hyperglycemia", "Healthcare Professional"
]
relationships = [
    ("Diabetes", "is associated with", "Blood Glucose regulation"),
    ("Pancreas", "produces", "Insulin"),
    ("Insulin", "helps regulate", "Blood Glucose"),
    ("Carbohydrates", "are broken down into", "Glucose"),
    ("Nutrition", "can influence", "Blood Glucose"),
    ("Physical Activity", "can affect", "Blood Glucose"),
    ("Glucose Monitoring", "measures", "Blood Glucose"),
    ("HbA1c", "reflects an estimate of", "average blood glucose over time"),
    ("Medication", "may be prescribed to help manage", "Diabetes"),
    ("Hypoglycemia", "means", "low Blood Glucose"),
    ("Hyperglycemia", "means", "high Blood Glucose"),
    ("Healthcare Professional", "individualises", "Medication and care plan")
]
print("Concept count:", len(concepts))
print("Relationship count:", len(relationships))
for item in relationships: print(" --".join(item))

## 5. NLP preprocessing
The preprocessing steps are lowercase conversion, tokenization, stopword removal and lemmatization.

In [ ]:
def preprocess_text(text):
    text = re.sub(r'[^a-z\\s]', ' ', str(text).lower())
    tokens = word_tokenize(text)
    tokens = [LEMMATIZER.lemmatize(t) for t in tokens if t not in STOP_WORDS and len(t) > 1]
    return ' '.join(tokens)

data['clean_query'] = data.query.apply(preprocess_text)
display(data[['query', 'clean_query', 'intent']].head(10))

## 6. TF-IDF and Logistic Regression classifier

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    data.clean_query, data.intent, test_size=0.25, random_state=42, stratify=data.intent
)
model = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)),
    ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced'))
])
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
metrics = {
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precision (weighted)': precision_score(y_test, y_pred, average='weighted', zero_division=0),
    'Recall (weighted)': recall_score(y_test, y_pred, average='weighted', zero_division=0),
    'F1-score (weighted)': f1_score(y_test, y_pred, average='weighted', zero_division=0)
}
display(pd.DataFrame(metrics.items(), columns=['Metric', 'Score']))
print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=model.classes_)
disp = ConfusionMatrixDisplay(cm, display_labels=model.classes_)
fig, ax = plt.subplots(figsize=(10, 7))
disp.plot(ax=ax, xticks_rotation=35, cmap='Blues', colorbar=False)
plt.title('Diabetes Assistant – Confusion Matrix')
plt.tight_layout()
plt.show()

## 7. Diabetes-domain knowledge base

In [ ]:
knowledge_base = {
    "general_information": (
        "Diabetes is a condition in which blood glucose levels are higher than normal because "
        "the body does not make enough insulin or cannot use insulin effectively. Common forms "
        "include type 1, type 2 and gestational diabetes. A healthcare professional can explain an individual's condition."
    ),
    "symptoms": (
        "Possible symptoms include increased thirst, frequent urination, fatigue, blurred vision "
        "and unexplained weight changes. Some people have few or no symptoms. Symptoms alone "
        "cannot confirm diabetes; testing and clinical assessment are needed."
    ),
    "nutrition": (
        "A balanced eating pattern can include vegetables, fibre-rich foods, suitable portions "
        "of carbohydrate-containing foods, lean protein and unsaturated fats. Individual needs "
        "vary; a registered dietitian or healthcare professional can help create a meal plan."
    ),
    "monitoring": (
        "Blood glucose meters measure glucose from a blood sample, while continuous glucose "
        "monitoring systems track glucose through a sensor. HbA1c estimates average blood glucose "
        "over roughly the previous 2–3 months. Ask your care team when and how often to test."
    ),
    "medication_and_treatment": (
        "Diabetes management may include nutrition, physical activity, monitoring and medication. "
        "Some people use insulin and others use different medicines. Only a qualified prescriber "
        "should recommend or change a medicine or dose. Do not stop prescribed treatment based on this assistant."
    ),
    "prevention_and_lifestyle": (
        "Regular physical activity, balanced nutrition, adequate sleep and avoiding tobacco can "
        "support overall metabolic health and may reduce the risk of type 2 diabetes. Discuss "
        "individual risks and prevention plans with a healthcare professional."
    )
}
for key, value in knowledge_base.items(): print(f'[{key}] {value}\\n')

## 8. Reasoning rules and integrated assistant

Rules implemented:
1. Emergency escalation has priority over classification.
2. Clearly unsupported healthcare topics are redirected.
3. Low-confidence classifications request clarification.
4. Supported intents retrieve an answer from the knowledge base.
5. Responses include a medical-information disclaimer.

The confidence threshold is a simple configurable heuristic, not a clinically validated score.

In [ ]:
EMERGENCY_KEYWORDS = [
    'unconscious', 'passed out', 'seizure', 'cannot breathe',
    'difficulty breathing', 'chest pain', 'unresponsive',
    'severe confusion', 'not waking'
]
DIABETES_SCOPE_KEYWORDS = [
    'diabetes', 'diabetic', 'blood sugar', 'blood glucose', 'glucose',
    'insulin', 'hba1c', 'a1c', 'hypoglycemia', 'hyperglycemia',
    'low sugar', 'high sugar', 'metformin', 'pancreas', 'carbohydrate'
]
CONFIDENCE_THRESHOLD = 0.28

def assistant_response(query):
    original = str(query).strip()
    lowered = original.lower()
    if not original:
        return "Please enter a question about diabetes care."

    # Rule 1: emergency escalation
    if any(term in lowered for term in EMERGENCY_KEYWORDS):
        return ("EMERGENCY ALERT: These symptoms may indicate a medical emergency. "
                "Call your local emergency number or seek emergency medical care now. "
                "Do not wait for an online assistant to assess the situation.")

    # Rule 2: unsupported-query handling
    if not any(term in lowered for term in DIABETES_SCOPE_KEYWORDS):
        return ("I currently answer questions within the diabetes-care information domain. "
                "Ask about diabetes, symptoms, nutrition, glucose monitoring, treatment or lifestyle. "
                "For other health concerns, contact a qualified healthcare professional.")

    cleaned = preprocess_text(original)
    probabilities = model.predict_proba([cleaned])[0]
    index = int(np.argmax(probabilities))
    intent = model.classes_[index]
    confidence = float(probabilities[index])

    # Rule 3: low-confidence handling
    if confidence < CONFIDENCE_THRESHOLD:
        return ("I am not confident enough to identify the intent. Please rephrase or ask a "
                "more specific diabetes-care question. Contact a healthcare professional for personal advice.")

    # Rule 4: knowledge retrieval
    answer = knowledge_base.get(intent)
    if answer is None:
        return "That topic is not currently supported by my knowledge base."

    # Rule 5: safety reminder
    return (f"Identified intent: {intent} (confidence: {confidence:.2f})\\n\\n"
            f"{answer}\\n\\n"
            "Reminder: General educational information only; not a diagnosis or treatment plan.")

print("Assistant is ready.")

## 9. Required test cases

In [ ]:
test_queries = [
    ("General information", "What is diabetes and how does it affect glucose?"),
    ("Symptoms", "Why am I thirsty and urinating frequently?"),
    ("Nutrition", "What kind of meals are useful for someone with diabetes?"),
    ("Monitoring", "What does an A1C test tell me?"),
    ("Medication and treatment", "Why might insulin be part of diabetes treatment?"),
    ("Prevention and lifestyle", "Can daily walking help reduce type 2 diabetes risk?"),
    ("Differently worded query", "My sugar readings have been elevated; what does that mean?"),
    ("Unsupported healthcare query", "How can I treat a broken ankle?"),
    ("Low-confidence query", "diabetes glucose pancreas relationship question"),
    ("Emergency query", "A person with diabetes is unconscious and having a seizure.")
]
for label, query in test_queries:
    print('=' * 85, '\\nTEST:', label, '\\nUSER:', query)
    print('ASSISTANT:', assistant_response(query), '\\n')

## 10. Interactive multi-query interface for Google Colab

Run `ask_assistant()` to chat with the assistant. Type `exit` to finish. Multiple queries are accepted in one session.

In [ ]:
conversation_history = []

def ask_assistant():
    print("Diabetes Care Information Assistant | type 'exit' to finish")
    while True:
        query = input("\\nYou: ").strip()
        if query.lower() in {'exit', 'quit', 'q'}:
            print("Assistant: Take care. Consult your healthcare professional for personal medical guidance.")
            break
        response = assistant_response(query)
        print("\\nAssistant:", response)
        conversation_history.append({'query': query, 'response': response})


 ask_assistant()


 display(pd.DataFrame(conversation_history))

## 11. Quick single-query test

In [ ]:
print(assistant_response("How can I monitor my blood glucose at home?"))